In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelBinarizer
import numpy as np

file_path = "BMW sales data (2010-2024).csv"
df = pd.read_csv(file_path)

df = df.drop(columns=['Mileage_KM', 'Sales_Volume'])
X = df.drop('Sales_Classification', axis=1)
y = df['Sales_Classification']
X_encoded = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

k = 5
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
y_pred_proba = knn.predict_proba(X_test_scaled)
lb = LabelBinarizer()
y_test_binary = lb.fit_transform(y_test)
y_scores = y_pred_proba[:, 1]

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label='High')
roc_auc = roc_auc_score(y_test_binary, y_scores)

results = pd.DataFrame({
    'Metric': ['Test Set Size', 'Accuracy', 'F1 Score', 'ROC-AUC'],
    'Value': [len(y_test), accuracy, f1, roc_auc],
    'Interpretation': [
        'Number of data points used to evaluate the model.',
        'Proportion of all classifications (High/Low) correctly predicted.',
        'Harmonic mean of precision and recall for the "High" sales class.',
        'Model\'s ability to distinguish between High and Low sales classes (higher is better).'
    ]
})

results['Value'] = results['Value'].apply(lambda x: f'{x:.4f}' if isinstance(x, (int, float, np.float64)) else x)
print(f"K-Nearest Neighbors Classification Results (K={k})\n")
print(results.to_string(index=False))

K-Nearest Neighbors Classification Results (K=5)

       Metric      Value                                                                        Interpretation
Test Set Size 10000.0000                                     Number of data points used to evaluate the model.
     Accuracy     0.6241                     Proportion of all classifications (High/Low) correctly predicted.
     F1 Score     0.2055                     Harmonic mean of precision and recall for the "High" sales class.
      ROC-AUC     0.4906 Model's ability to distinguish between High and Low sales classes (higher is better).
